In [ ]:
# --- Paths (run this cell first) ---------------------------------------------------------
# Shared schema.org assets vs. per-article files now live in separate folders. Switch
# ARTICLE_DIR to work on a different article; SCHEMA_DIR rarely changes.
from pathlib import Path

SCHEMA_DIR = Path("schema_org_files")     # schemaorg dump, types/properties files (shared)
ARTICLE_DIR = Path("politics_article")    # article.txt, entities_*, typed_entities, triples (per-article)
ARTICLE_DIR.mkdir(exist_ok=True)
SCHEMA_DIR.mkdir(exist_ok=True)
print("SCHEMA_DIR =", SCHEMA_DIR, "| ARTICLE_DIR =", ARTICLE_DIR)

In [ ]:
# get text from a news article and save it to a file
import requests
from bs4 import BeautifulSoup
def get_article_text(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    paragraphs = soup.find_all('p')
    article_text = '\n'.join([p.get_text() for p in paragraphs])
    return article_text
if __name__ == "__main__":    
    url = input("Enter the URL of the news article: ")
    article_text = get_article_text(url)
    with open(ARTICLE_DIR / 'article.txt', 'w') as file:        
        file.write(article_text)
    print(f"Article text has been saved to {ARTICLE_DIR / 'article.txt'}")

# https://www.bbc.com/news/articles/c7vyn17g832o

## Entity & triple extraction with an open-source LLM (Ollama)

One-time setup (run in a terminal, not here):

```bash
# 1. Install the Ollama runtime (macOS)
brew install ollama          # or download from https://ollama.com/download

# 2. Start the local server (keep it running in its own terminal)
ollama serve

# 3. Pull an open-source model
ollama pull llama3.1         # ~4.7 GB; or try mistral / qwen2.5

# 4. Install the Python client (used by the cells below)
/usr/local/bin/python -m pip install ollama
```


In [ ]:
import json
from collections import defaultdict

with open(SCHEMA_DIR / "schemaorg-current-https.jsonld") as f:
    data = json.load(f)


def as_list(x):
    """Normalise a field that may be missing, a single value, or a list."""
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def short(ref):
    """Turn {'@id': 'schema:MediaObject'} (or 'schema:MediaObject') into 'MediaObject'."""
    if isinstance(ref, dict):
        ref = ref.get("@id", "")
    return ref.split(":")[-1]


# Collect every class and its parent class(es) via rdfs:subClassOf
children = defaultdict(list)   # parent -> [child, child, ...]
parents = defaultdict(list)    # child  -> [parent, ...]
types = []

for item in data["@graph"]:
    item_type = item.get("@type")
    is_class = item_type == "rdfs:Class" or (
        isinstance(item_type, list) and "rdfs:Class" in item_type
    )
    if not is_class:
        continue

    name = item.get("rdfs:label")
    if isinstance(name, dict):
        name = name.get("@value")
    if isinstance(name, list):
        name = name[0]
    if not name:
        continue

    types.append(name)
    for parent_ref in as_list(item.get("rdfs:subClassOf")):
        parent = short(parent_ref)
        if parent:
            children[parent].append(name)
            parents[name].append(parent)

# Roots = classes that have no parent (e.g. "Thing")
roots = sorted(t for t in types if not parents.get(t))

print(f"Found {len(types)} types | {len(roots)} root(s): {roots}")

In [ ]:
# Save as an indented tree for entity typing: shows granularity, each type once, token-light.
# Place each class under its first parent that is itself a defined type (a few schema.org
# parents like "Document"/"LegalDocument" are referenced but not defined here, so we skip them).
typeset = set(types)
primary_children = defaultdict(list)
extra_roots = []
for t in types:
    parent = next((p for p in parents.get(t, []) if p in typeset), None)
    if parent:
        primary_children[parent].append(t)
    elif t not in roots:
        extra_roots.append(t)   # parent(s) undefined in this file -> treat as its own root


def walk(node, depth, seen, lines):
    lines.append("  " * depth + node)          # 2 spaces per level
    if node in seen:
        return
    seen.add(node)
    for child in sorted(set(primary_children.get(node, []))):
        walk(child, depth + 1, seen, lines)


lines, seen = [], set()
for root in roots + sorted(extra_roots):
    walk(root, 0, seen, lines)

with open(SCHEMA_DIR / "types.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

assert len(seen) == len(types), f"dropped {len(types) - len(seen)} types!"
print(f"Wrote {len(lines)} lines covering all {len(types)} types")
print("\n".join(lines[:25]))

In [ ]:
# Extract all schema.org properties with subject (domain) and object (range) type constraints.
# schema.org uses schema:domainIncludes / schema:rangeIncludes (not rdfs:domain/range),
# meaning a property can apply to multiple domain types and accept multiple range types.

import json
from collections import defaultdict

with open(SCHEMA_DIR / "schemaorg-current-https.jsonld") as f:
    data = json.load(f)

def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def short(ref):
    if isinstance(ref, dict):
        ref = ref.get("@id", "")
    return ref.split(":")[-1]

properties = []
by_domain = defaultdict(list)   # domain_type -> [{property, range}]

for item in data["@graph"]:
    item_type = item.get("@type")
    is_property = item_type == "rdf:Property" or (
        isinstance(item_type, list) and "rdf:Property" in item_type
    )
    if not is_property:
        continue

    name = item.get("rdfs:label")
    if isinstance(name, dict):
        name = name.get("@value")
    if isinstance(name, list):
        name = name[0]
    if not name:
        continue

    domain = sorted({short(d) for d in as_list(item.get("schema:domainIncludes"))})
    range_ = sorted({short(r) for r in as_list(item.get("schema:rangeIncludes"))})

    entry = {"property": name, "domain": domain, "range": range_}
    properties.append(entry)
    for d in domain:
        by_domain[d].append({"property": name, "range": range_})

properties.sort(key=lambda p: p["property"])

# --- flat JSON (all properties) ---
with open(SCHEMA_DIR / "properties.json", "w") as f:
    json.dump(properties, f, indent=2, ensure_ascii=False)

# --- flat text for prompt use ---
# Format: propertyName | domain: A, B | range: X, Y
prop_lines = []
for p in properties:
    d_str = ", ".join(p["domain"]) if p["domain"] else "ANY"
    r_str = ", ".join(p["range"])  if p["range"]  else "ANY"
    prop_lines.append(f"{p['property']} | domain: {d_str} | range: {r_str}")

with open(SCHEMA_DIR / "properties.txt", "w") as f:
    f.write("\n".join(prop_lines) + "\n")

# --- grouped by domain type (useful when you know the subject type) ---
grouped = {dtype: sorted(props, key=lambda x: x["property"])
           for dtype, props in sorted(by_domain.items())}
with open(SCHEMA_DIR / "properties_by_domain.json", "w") as f:
    json.dump(grouped, f, indent=2, ensure_ascii=False)

print(f"Found {len(properties)} properties across {len(by_domain)} domain types")
print(f"Wrote: properties.json  |  properties.txt  |  properties_by_domain.json")
print("\nSample (first 8 properties):")
for p in properties[:8]:
    d_str = ", ".join(p["domain"]) if p["domain"] else "ANY"
    r_str = ", ".join(p["range"])  if p["range"]  else "ANY"
    print(f"  {p['property']:30s}  domain: {d_str:40s}  range: {r_str}")

In [ ]:
# Build an indented property hierarchy using rdfs:subPropertyOf,
# mirroring the type hierarchy in types.txt.

import json
from collections import defaultdict

with open(SCHEMA_DIR / "schemaorg-current-https.jsonld") as f:
    data = json.load(f)

def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def short(ref):
    if isinstance(ref, dict):
        ref = ref.get("@id", "")
    return ref.split(":")[-1]

prop_children = defaultdict(list)   # parent -> [child, ...]
prop_parents  = defaultdict(list)   # child  -> [parent, ...]
prop_names = []

for item in data["@graph"]:
    item_type = item.get("@type")
    is_property = item_type == "rdf:Property" or (
        isinstance(item_type, list) and "rdf:Property" in item_type
    )
    if not is_property:
        continue

    name = item.get("rdfs:label")
    if isinstance(name, dict):
        name = name.get("@value")
    if isinstance(name, list):
        name = name[0]
    if not name:
        continue

    prop_names.append(name)
    for parent_ref in as_list(item.get("rdfs:subPropertyOf")):
        parent = short(parent_ref)
        if parent:
            prop_children[parent].append(name)
            prop_parents[name].append(parent)

prop_set = set(prop_names)

# Roots = properties with no parent defined in the file
prop_roots = sorted(p for p in prop_names if not prop_parents.get(p))

# Properties whose listed parent isn't itself in the file → treat as extra roots
extra_prop_roots = []
primary_prop_children = defaultdict(list)
for p in prop_names:
    parent = next((par for par in prop_parents.get(p, []) if par in prop_set), None)
    if parent:
        primary_prop_children[parent].append(p)
    elif p not in prop_roots:
        extra_prop_roots.append(p)

def walk_props(node, depth, seen, lines):
    lines.append("  " * depth + node)
    if node in seen:
        return
    seen.add(node)
    for child in sorted(set(primary_prop_children.get(node, []))):
        walk_props(child, depth + 1, seen, lines)

lines, seen = [], set()
for root in prop_roots + sorted(extra_prop_roots):
    walk_props(root, 0, seen, lines)

with open(SCHEMA_DIR / "property_hierarchy.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

assert len(seen) == len(prop_names), f"dropped {len(prop_names) - len(seen)} properties!"
print(f"Found {len(prop_names)} properties | {len(prop_roots)} root(s)")
print(f"Wrote property_hierarchy.txt ({len(lines)} lines)")
print("\nFirst 20 lines:")
print("\n".join(lines[:20]))

In [ ]:
# Entity recognition + hierarchical typing: assign the MOST SPECIFIC schema.org type.
# Calls Ollama via its REST API (no extra pip package needed) with a large context window
# so the ~10k-token type hierarchy is NOT truncated.
import json
import requests
from IPython.display import Markdown, display

#MODEL = "llama3.1"
MODEL = "llama3.2:3b"

OLLAMA_URL = "http://localhost:11434/api/chat"

with open(SCHEMA_DIR / "types.txt") as f:
    type_hierarchy = f.read()          # indented tree: indentation = subtype-of

with open(ARTICLE_DIR / "article.txt") as f:
    article_text = f.read()

TYPING_SYSTEM_PROMPT = """You are an expert Entity Recognition and Typing system.

You are given:
1. A TEXT to analyze.
2. A TYPE HIERARCHY: the allowed entity types (from the schema.org vocabulary), written as an indented tree. Indentation encodes subtype relationships: a more-indented type is a more specific kind of the less-indented type directly above it. For example
    Place
      CivicStructure
        Airport
means an Airport is a kind of CivicStructure, which is a kind of Place.

TASK: Extract every entity explicitly mentioned in the TEXT and assign each the MOST SPECIFIC type in the hierarchy that the text justifies.

RULES:
1. Extract entities exactly as they appear; preserve original casing and punctuation. Do not invent, infer, or normalize entities that are not explicitly mentioned.
2. Use ONLY type names from the hierarchy (the label only, ignore the indentation whitespace).
3. Assign the DEEPEST / most specific type the evidence supports. Walk down the tree as far as the text justifies, but no further: if the text only supports a general type, use the general type rather than guessing a subtype.
4. If an entity matches no type in the hierarchy, assign "OTHER".
5. Include every distinct entity only once (deduplicate repeated mentions of the same entity).
6. "confidence" is your certainty in the assigned type, between 0 and 1.
7. Return ONLY valid JSON matching the schema below. No explanations, no extra text.

OUTPUT SCHEMA:
{ "entities": [ { "text": "entity mention", "type": "entity_type", "confidence": 0.95 } ] }
"""

user_message = f"TYPE HIERARCHY:\n{type_hierarchy}\n\nTEXT:\n{article_text}"

est_tokens = (len(type_hierarchy) + len(article_text)) // 4
print(f"~{est_tokens} prompt tokens (types + article); calling {MODEL} ...")

payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": TYPING_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ],
    "stream": False,
    "format": "json",
    "options": {"temperature": 0, "num_ctx": 8192, "num_predict": -1},   # num_predict=-1 = unlimited output
}
response = requests.post(OLLAMA_URL, json=payload, timeout=2000)
response.raise_for_status()

print(f"Prompt tokens used:  {response.json().get('prompt_eval_count')}")
print(f"Output tokens generated: {response.json().get('eval_count')}")

typed = json.loads(response.json()["message"]["content"])

# Save machine-readable output
with open(ARTICLE_DIR / "typed_entities.json", "w") as f:
    json.dump(typed, f, indent=2, ensure_ascii=False)

# Display entities and their types in bold
ents = typed.get("entities", [])
md = [f"**Found {len(ents)} entities**\n"]
for e in ents:
    md.append(f"- **{e.get('text','')}** → **{e.get('type','')}**  _(conf {e.get('confidence','?')})_")
display(Markdown("\n".join(md)))

In [ ]:
# Triple extraction (second pass): takes the typed entities from the previous cell + the
# schema.org property list (with domain/range constraints) and asks the LLM to extract
# triples strictly between the already-extracted entities.

import json
import requests
from IPython.display import Markdown, display

MODEL = "llama3.1"
OLLAMA_URL = "http://localhost:11434/api/chat"

with open(ARTICLE_DIR / "article.txt") as f:
    article_text = f.read()

with open(ARTICLE_DIR / "typed_entities.json") as f:
    entities_obj = json.load(f)
entities = entities_obj.get("entities", [])

with open(SCHEMA_DIR / "properties.txt") as f:
    properties_text = f.read()

# Compact entity view for the prompt: drop confidence, keep only text+type
entities_for_prompt = [{"text": e["text"], "type": e["type"]} for e in entities]
entities_json = json.dumps(entities_for_prompt, ensure_ascii=False, indent=2)

TRIPLE_SYSTEM_PROMPT = """You are an expert Knowledge Graph Triple Extraction system.

You are given:
1. A TEXT to analyze.
2. A list of ENTITIES already extracted from the TEXT, each with its schema.org type.
3. A list of schema.org PROPERTIES, one per line, in the format:
       propertyName | domain: SubjectType(s) | range: ObjectType(s)
   - "domain" lists the schema.org types valid as the SUBJECT of the property.
   - "range"  lists the schema.org types valid as the OBJECT of the property.
   - "ANY" means no restriction.

TASK: Extract every triple (subject, property, object) that the TEXT explicitly states or strongly implies.

RULES:
1. Both the SUBJECT and the OBJECT of every triple MUST be entities from the ENTITIES list.
   Use the entity's EXACT "text" value. Do NOT introduce any new mention not in the list.
2. The PROPERTY MUST be a propertyName from the PROPERTIES list (use the label only, no pipes).
3. Respect domain/range constraints: the subject's type should match (or be a subtype of) one of
   the property's domain types, and the object's type should match (or be a subtype of) one of
   the property's range types. Use schema.org subtype knowledge (e.g. SportsTeam is an Organization).
4. Extract only triples explicitly supported by the TEXT. Do not invent relationships.
5. Deduplicate identical triples.
6. "confidence" is your certainty in the triple, between 0 and 1.
7. Return ONLY valid JSON matching the schema below. No explanations, no extra text.

OUTPUT SCHEMA:
{ "triples": [ { "subject": "entity text", "property": "propertyName", "object": "entity text", "confidence": 0.95 } ] }
"""

user_message = (
    f"ENTITIES:\n{entities_json}\n\n"
    f"PROPERTIES:\n{properties_text}\n\n"
    f"TEXT:\n{article_text}"
)

est_tokens = (len(properties_text) + len(article_text) + len(entities_json)) // 4
print(f"~{est_tokens} prompt tokens (entities + properties + article); calling {MODEL} ...")

payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": TRIPLE_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ],
    "stream": False,
    "format": "json",
    "options": {"temperature": 0, "num_ctx": 16384, "num_predict": -1},
}
response = requests.post(OLLAMA_URL, json=payload, timeout=2000)
response.raise_for_status()

print(f"Prompt tokens used:      {response.json().get('prompt_eval_count')}")
print(f"Output tokens generated: {response.json().get('eval_count')}")

triples_obj = json.loads(response.json()["message"]["content"])
triples = triples_obj.get("triples", [])

# Post-hoc validation: every subject/object must appear in the entity list
entity_texts = {e["text"] for e in entities}
valid, invalid = [], []
for t in triples:
    if t.get("subject") in entity_texts and t.get("object") in entity_texts:
        valid.append(t)
    else:
        invalid.append(t)

with open(ARTICLE_DIR / "triples.json", "w") as f:
    json.dump({"triples": valid, "invalid": invalid}, f, indent=2, ensure_ascii=False)

# Display
md = [f"**Found {len(triples)} triples** ({len(valid)} valid, {len(invalid)} invalid)\n"]
for t in valid:
    md.append(f"- **{t.get('subject','')}** — *{t.get('property','')}* → **{t.get('object','')}**  _(conf {t.get('confidence','?')})_")
if invalid:
    md.append("\n**Rejected (subject or object not in entity list):**")
    for t in invalid:
        md.append(f"- ~~{t.get('subject','')} — {t.get('property','')} → {t.get('object','')}~~")
display(Markdown("\n".join(md)))

check against the one i got in claude and chatgpt

In [ ]:
# Compare entities extracted by ChatGPT vs Claude vs llama3.2:3b.
# Matches entities by text (case-insensitive). Generalised to N models. Produces:
#   - entities_all.json          : every entity with each model's type + which models found it
#   - entities_common.json       : entities found by ALL models (with per-model type + agreement)
#   - entities_only_<model>.json : entities found by exactly one model

import json
from itertools import combinations

# model label -> source file (inside ARTICLE_DIR)
MODELS = {
    "chatgpt": "entities_chatgpt.json",
    "claude":  "entities_claude.json",
    "llama":   "entities_llama3.2:3b.json",
}

def index_by_text(ents):
    # keep first occurrence; key on lowercased text for case-insensitive match
    out = {}
    for e in ents:
        key = e["text"].strip().lower()
        if key and key not in out:
            out[key] = e
    return out

# Load + index each model's entities
idx = {}
for name, path in MODELS.items():
    with open(ARTICLE_DIR / path) as f:
        idx[name] = index_by_text(json.load(f).get("entities", []))

all_keys = set().union(*idx.values())

# --- Per-entity view across all models ---
all_entities = []
for k in sorted(all_keys):
    found_in = [m for m in MODELS if k in idx[m]]
    types = {m: idx[m][k]["type"] for m in found_in}
    # representative original casing: first model (by MODELS order) that has it
    text = next(idx[m][k]["text"] for m in MODELS if k in idx[m])
    all_entities.append({
        "text": text,
        "found_in": found_in,
        "n_models": len(found_in),
        "types": types,
        "types_agree": len(set(types.values())) == 1,
    })

# --- Common to ALL models ---
common_keys = set.intersection(*[set(idx[m]) for m in MODELS])
common = [e for e in all_entities if e["n_models"] == len(MODELS)]
type_agreements    = sum(1 for e in common if e["types_agree"])
type_disagreements = len(common) - type_agreements

# --- Found by exactly one model ---
only = {m: [e for e in all_entities if e["found_in"] == [m]] for m in MODELS}

# --- Write files (into ARTICLE_DIR) ---
with open(ARTICLE_DIR / "entities_all.json", "w") as f:
    json.dump({"entities": all_entities}, f, indent=2, ensure_ascii=False)
with open(ARTICLE_DIR / "entities_common.json", "w") as f:
    json.dump({"entities": common}, f, indent=2, ensure_ascii=False)
for m in MODELS:
    safe = m.replace(":", "_").replace("/", "_")
    with open(ARTICLE_DIR / f"entities_only_{safe}.json", "w") as f:
        json.dump({"entities": only[m]}, f, indent=2, ensure_ascii=False)

# --- Summary ---
for m in MODELS:
    print(f"{m:8s} entities: {len(idx[m])}")
print(f"\nCommon to all {len(MODELS)}: {len(common)}  "
      f"(type agrees on {type_agreements}, disagrees on {type_disagreements})")

print("\nPairwise overlap:")
for a, b in combinations(MODELS, 2):
    print(f"  {a} ∩ {b}: {len(set(idx[a]) & set(idx[b]))}")

print("\nUnique to a single model:")
for m in MODELS:
    print(f"  only {m}: {len(only[m])}")

# Type disagreements among models that share an entity (2+ models, not unanimous)
disagree = [e for e in all_entities if e["n_models"] >= 2 and not e["types_agree"]]
if disagree:
    print("\nType disagreements (entities seen by 2+ models with differing types):")
    for e in disagree:
        types_str = ", ".join(f"{m}={e['types'][m]}" for m in e["found_in"])
        print(f"  {e['text']:40s} {types_str}")

prompt of OIE of entites

You are an expert Entity Recognition and Typing system.

You are given a TEXT to analyze.

TASK: Extract every entity explicitly mentioned in the TEXT. For each entity, assign the single most appropriate type label, choosing the label freely based on what the entity is. You are NOT given a fixed set of types: invent whatever type label best describes each entity. After extraction, produce a definition for every unique type label you used.

RULES:
1. Extract entities exactly as they appear; preserve original casing and punctuation. Do not invent, infer, or normalize entities that are not explicitly mentioned.
2. Assign each entity a free-form type label of your own choosing. Pick the label that most precisely captures what kind of thing the entity is, given the evidence in the text. Do not restrict yourself to any predefined vocabulary.
3. Keep type labels consistent: if two entities are the same kind of thing, give them the exact same label (same spelling, same casing). Prefer a single clear noun or short noun phrase per label (e.g. "Person", "City", "Software Library", "Legal Case").
4. Choose the most specific label the text justifies. Use a general label only when the text does not support a more specific one; do not guess a specificity the evidence does not warrant.
5. Include every distinct entity only once (deduplicate repeated mentions of the same entity).
6. "confidence" is your certainty in the assigned type, between 0 and 1.
7. After extracting all entities, collect every UNIQUE type label you used and write a concise one-sentence definition for each. The definition should describe the category in general terms (what this kind of thing is), independent of the specific entities in the text.
8. The set of distinct type labels used in "entities" must be exactly the set of labels defined in "types": every label used on an entity has exactly one definition, and every defined label is used by at least one entity. A label may be used by multiple entities, but is defined only once.
9. Return ONLY valid JSON matching the schema below. No explanations, no extra text.

OUTPUT SCHEMA:
{
  "entities": [
    { "text": "entity mention", "type": "type_label", "confidence": 0.95 }
  ],
  "types": [
    { "type": "type_label", "definition": "A concise general definition of this type." }
  ]
}

allowing more than one type for same mention (like country and national football team)

You are an expert Entity Recognition and Typing system.
You are given a TEXT to analyze.
TASK: Extract every entity explicitly mentioned in the TEXT. For each entity, assign the single most appropriate type label, choosing the label freely based on what the entity is. You are NOT given a fixed set of types: invent whatever type label best describes each entity. The SAME surface text may be extracted more than once if, in different parts of the text, it clearly refers to different kinds of things (i.e. warrants different type labels). After extraction, produce a definition for every unique type label you used.
RULES:
1. Extract entities exactly as they appear; preserve original casing and punctuation. Do not invent, infer, or normalize entities that are not explicitly mentioned.
2. Assign each entity a free-form type label of your own choosing. Pick the label that most precisely captures what kind of thing the entity is, given the evidence in the text. Do not restrict yourself to any predefined vocabulary.
3. Keep type labels consistent: if two entities are the same kind of thing, give them the exact same label (same spelling, same casing). Prefer a single clear noun or short noun phrase per label (e.g. "Person", "City", "Software Library", "Legal Case").
4. Choose the most specific label the text justifies. Use a general label only when the text does not support a more specific one; do not guess a specificity the evidence does not warrant.
5. Deduplicate on the (text, type) pair, not on text alone:
   - If the same surface text appears multiple times AND refers to the same kind of thing each time, include it ONCE.
   - If the same surface text refers to genuinely different kinds of things in different places (each supported by the text), include it ONCE PER DISTINCT TYPE, using the same surface text with the different type labels. Only do this when the evidence clearly supports more than one type for that exact mention text; do not split a single referent across types just because a label is uncertain.
6. "confidence" is your certainty in the assigned type, between 0 and 1.
7. After extracting all entities, collect every UNIQUE type label you used and write a concise one-sentence definition for each. The definition should describe the category in general terms (what this kind of thing is), independent of the specific entities in the text.
8. The set of distinct type labels used in "entities" must be exactly the set of labels defined in "types": every label used on an entity has exactly one definition, and every defined label is used by at least one entity. A label may be used by multiple entities, but is defined only once. The same (text, type) pair must not appear more than once.
9. Return ONLY valid JSON matching the schema below. No explanations, no extra text.
OUTPUT SCHEMA:
{
  "entities": [
    { "text": "entity mention", "type": "type_label", "confidence": 0.95 }
  ],
  "types": [
    { "type": "type_label", "definition": "A concise general definition of this type." }
  ]
}

with passing schema.org type hirerachy to match where relevant

You are an expert Entity Recognition and Typing system.
You are given a TEXT to analyze and a SCHEMA.ORG TYPE HIERARCHY to type against.

SCHEMA.ORG TYPE HIERARCHY:
The attached hierarchy is a fragment of the schema.org vocabulary, written as an indented tree. Indentation encodes subtype relationships: a more-indented type is a more specific kind of the less-indented type directly above it. For example:
    Place
      CivicStructure
        Airport
means an Airport is a kind of CivicStructure, which is a kind of Place. Treat the attached hierarchy as the authoritative list of available schema.org types.

TASK: Extract every entity explicitly mentioned in the TEXT. For each entity, assign the single most appropriate type label. Decide the type label as follows:
- If a type in the attached SCHEMA.ORG TYPE HIERARCHY is a highly relevant and specific match for the entity, use that schema.org type (exact schema.org name and casing, e.g. "Person", "Organization", "NewsArticle").
- If no schema.org type is a good fit, or the only honest schema.org match is too generic to be informative when a more precise label is available, invent a free-form type label that best describes the entity, exactly as in open information extraction.
The SAME surface text may be extracted more than once if, in different parts of the text, it clearly refers to different kinds of things (i.e. warrants different type labels). After extraction, produce a definition for every unique type label you used.

RULES:
1. Extract entities exactly as they appear; preserve original casing and punctuation. Do not invent, infer, or normalize entities that are not explicitly mentioned.

2. Assign each entity a single type label. Prefer a schema.org type from the attached hierarchy when it is a highly relevant and specific match for the entity. Use a free-form label of your own choosing only when no schema.org type genuinely fits, or when every fitting schema.org type is so broad that a free-form label would be substantially more precise.

3. "Highly relevant and specific match" means the entity is clearly an instance of that schema.org type given the evidence in the text, and the type is reasonably specific rather than a loose or generic approximation. Do not force an entity into a schema.org type when the fit is only approximate. When in doubt between a generic schema.org type and a precise free-form label, choose whichever the text most accurately justifies; do not default to schema.org just because a broad parent type technically applies.

4. Choose the most specific label the text justifies, whether schema.org or free-form. Within schema.org, prefer the deepest (most indented) type that is clearly supported by the text over its broader parents. Use a general label only when the text does not support a more specific one; do not guess a specificity the evidence does not warrant.

5. Keep type labels consistent: if two entities are the same kind of thing, give them the exact same label (same spelling, same casing). For schema.org types, use the exact name as it appears in the attached hierarchy. For free-form types, prefer a single clear noun or short noun phrase per label (e.g. "Software Library", "Legal Case").

6. Record the source of each type with "type_source":
- "schema.org" if the type label is taken from the attached SCHEMA.ORG TYPE HIERARCHY.
- "open" if the type label is a free-form label you invented.
The same surface text using the same type label must always carry the same "type_source".

7. Deduplicate on the (text, type) pair, not on text alone:
- If the same surface text appears multiple times AND refers to the same kind of thing each time, include it ONCE.
- If the same surface text refers to genuinely different kinds of things in different places (each supported by the text), include it ONCE PER DISTINCT TYPE, using the same surface text with the different type labels. Only do this when the evidence clearly supports more than one type for that exact mention text; do not split a single referent across types just because a label is uncertain.

8. "confidence" is your certainty in the assigned type, between 0 and 1.

9. After extracting all entities, collect every UNIQUE type label you used and write a concise one-sentence definition for each. For schema.org types, the definition should reflect the meaning of that type in the schema.org vocabulary. For free-form types, the definition should describe the category in general terms (what this kind of thing is). In both cases the definition is independent of the specific entities in the text.

10. The set of distinct type labels used in "entities" must be exactly the set of labels defined in "types": every label used on an entity has exactly one definition, and every defined label is used by at least one entity. A label may be used by multiple entities, but is defined only once. The same (text, type) pair must not appear more than once. Each defined type must carry the same "type_source" that was used for it on the entities.

11. Return ONLY valid JSON matching the schema below. No explanations, no extra text.

OUTPUT SCHEMA:
{
  "entities": [
    { "text": "entity mention", "type": "type_label", "type_source": "schema.org", "confidence": 0.95 }
  ],
  "types": [
    { "type": "type_label", "type_source": "schema.org", "definition": "A concise general definition of this type." }
  ]
}

prompt for OIE relations

You are an expert Knowledge Graph Triple Extraction system.

You are given:
1. A TEXT to analyze.
2. A list of ENTITIES already extracted from the TEXT, each with its type.

TASK: Extract every triple (subject, relation, object) that the TEXT explicitly states or strongly implies. You are NOT given a fixed set of relations: name each relation freely based on what the TEXT expresses.

RULES:
1. Both the SUBJECT and the OBJECT of every triple MUST be entities from the ENTITIES list. Use the entity's EXACT "text" value. Do NOT introduce any new mention not in the list.
2. Name the relation with a free-form label of your own choosing. Pick the label that most precisely captures the relationship the TEXT expresses between subject and object. Do not restrict yourself to any predefined vocabulary.
3. Use a verb or verb phrase in lowerCamelCase, read as "subject RELATION object" (e.g. "foundedBy", "locatedIn", "worksFor", "marriedTo"). Keep the direction consistent with subject and object: the subject is the thing the relation is about, the object is what it points to.
4. Keep relation labels consistent: if two triples express the same kind of relationship, give them the exact same label (same spelling, same casing).
5. Extract only triples explicitly stated or strongly implied by the TEXT. Do not invent relationships that are not supported.
6. Deduplicate identical triples (same subject, relation, and object).
7. "confidence" is your certainty in the triple, between 0 and 1.
8. After extracting all triples, collect every UNIQUE relation label you used and write a concise one-sentence definition for each. The definition should describe the relationship in general terms (what it means for one thing to stand in this relation to another), independent of the specific entities in the text.
9. The set of distinct relation labels used in "triples" must be exactly the set of labels defined in "relations": every label used in a triple has exactly one definition, and every defined label is used by at least one triple. A label may be used by multiple triples, but is defined only once.
10. Return ONLY valid JSON matching the schema below. No explanations, no extra text.

OUTPUT SCHEMA:
{
  "triples": [
    { "subject": "entity text", "relation": "relationLabel", "object": "entity text", "confidence": 0.95 }
  ],
  "relations": [
    { "relation": "relationLabel", "definition": "A concise general definition of this relation." }
  ]
}

direct extraction

You are an expert Knowledge Graph Triple Extraction system.
You are given a TEXT to analyze.

TASK: Extract every (subject, relation, object) triple that the TEXT explicitly states or strongly implies. You are NOT given a fixed set of relations: name each relation freely based on what the TEXT expresses.

RULES:
1. Both the SUBJECT and the OBJECT of every triple must be entities explicitly mentioned in the TEXT. Extract them exactly as they appear; preserve original casing and punctuation. Do not invent, infer, or normalize mentions that are not explicitly in the TEXT.
2. Name the relation with a free-form label of your own choosing. Pick the label that most precisely captures the relationship the TEXT expresses between subject and object. Do not restrict yourself to any predefined vocabulary.
3. Use a verb or verb phrase in lowerCamelCase, read as "subject RELATION object" (e.g. "foundedBy", "locatedIn", "worksFor", "marriedTo"). Keep the direction consistent with subject and object: the subject is the thing the relation is about, the object is what it points to.
4. Keep relation labels consistent: if two triples express the same kind of relationship, give them the exact same label (same spelling, same casing).
5. Extract only triples explicitly stated or strongly implied by the TEXT. Do not invent relationships that are not supported.
6. Deduplicate identical triples (same subject, relation, and object).
7. "confidence" is your certainty in the triple, between 0 and 1.
8. After extracting all triples, collect every UNIQUE relation label you used and write a concise one-sentence definition for each. The definition should describe the relationship in general terms (what it means for one thing to stand in this relation to another), independent of the specific entities in the TEXT.
9. The set of distinct relation labels used in "triples" must be exactly the set of labels defined in "relations": every label used in a triple has exactly one definition, and every defined label is used by at least one triple. A label may be used by multiple triples, but is defined only once.
10. Return ONLY valid JSON matching the schema below. No explanations, no extra text.

OUTPUT SCHEMA:
{
  "triples": [
    { "subject": "entity text", "relation": "relationLabel", "object": "entity text", "confidence": 0.95 }
  ],
  "relations": [
    { "relation": "relationLabel", "definition": "A concise general definition of this relation." }
  ]
}

wikidata

In [15]:
import requests

ENDPOINT = "https://query.wikidata.org/sparql"
HEADERS = {"User-Agent": "schema-thesis-probe/0.1 (research)", "Accept": "application/sparql-results+json"}

def fetch(qids):
    """Get label, description, and aliases for a list of Wikidata Q/P ids."""
    values = " ".join(f"wd:{q}" for q in qids)
    query = f"""
    SELECT ?item ?label ?description
           (GROUP_CONCAT(DISTINCT ?alias; SEPARATOR=" | ") AS ?aliases)
    WHERE {{
      VALUES ?item {{ {values} }}
      OPTIONAL {{ ?item rdfs:label    ?label       FILTER(LANG(?label)="en") }}
      OPTIONAL {{ ?item schema:description ?description FILTER(LANG(?description)="en") }}
      OPTIONAL {{ ?item skos:altLabel  ?alias       FILTER(LANG(?alias)="en") }}
    }}
    GROUP BY ?item ?label ?description
    """
    r = requests.get(ENDPOINT, params={"query": query}, headers=HEADERS, timeout=30)
    r.raise_for_status()
    for row in r.json()["results"]["bindings"]:
        qid = row["item"]["value"].split("/")[-1]
        label = row.get("label", {}).get("value", "")
        desc  = row.get("description", {}).get("value", "")
        alias = row.get("aliases", {}).get("value", "")
        print(f"\n{qid}  {label}\n  def: {desc}\n  aliases: {alias}")

# A few your project actually cares about:
#   Q2380335 airstrike | Q1656682 event | P825 target | P828 has cause | Q11473 missile
fetch(["Q2380335", "Q1656682", "P825", "P828", "Q11473"])


P825  dedicated to
  def: person or organization to whom the subject was dedicated
  aliases: dedication | dedicatee | tribute to | devoted to

P828  has cause
  def: underlying cause, entity that ultimately resulted in this effect
  aliases: cause | reason | because | why | due to | implied by | caused by | effect of | had cause | had reason | had underlying cause | has causes | has reason | has underlying cause | initial cause | originated due to | outcome of | result of | start cause | ultimate cause | ultimate causes | underlying cause

Q1656682  event
  def: temporary and scheduled happening, like a conference, festival, competition or similar
  aliases: activity | happening | events

Q11473  thermodynamics
  def: branch of physics concerned with heat, work, temperature, and thermal or internal energy
  aliases: pyro-dynamics | thermal science | thermo-dynamics

Q2380335  airstrike
  def: event in which aircraft are used to attack targets on land or at sea
  aliases: strike | air

In [16]:
def top_properties_for_class(qid, limit=15):
    query = f"""
    SELECT ?p ?pLabel (COUNT(*) AS ?n) WHERE {{
      ?item wdt:P31/wdt:P279* wd:{qid} .       # instances (incl. subclasses) of the class
      ?item ?prop ?val .
      ?p wikibase:directClaim ?prop ; rdfs:label ?pLabel . FILTER(LANG(?pLabel)="en")
    }} GROUP BY ?p ?pLabel ORDER BY DESC(?n) LIMIT {limit}
    """
    r = requests.get(ENDPOINT, params={"query": query}, headers=HEADERS, timeout=60)
    r.raise_for_status()
    for row in r.json()["results"]["bindings"]:
        print(f'{int(row["n"]["value"]):>6}  {row["p"]["value"].split("/")[-1]:8} {row["pLabel"]["value"]}')

top_properties_for_class("Q1656682")   # properties commonly used on "event" instances

ReadTimeout: HTTPSConnectionPool(host='query.wikidata.org', port=443): Read timed out. (read timeout=60)

In [17]:
def top_properties_for_class(qid, sample=300, limit=15):
    query = f"""
    SELECT ?p ?pLabel (COUNT(*) AS ?n) WHERE {{
      {{ SELECT ?item WHERE {{ ?item wdt:P31 wd:{qid} . }} LIMIT {sample} }}
      ?item ?prop ?val .
      ?p wikibase:directClaim ?prop ; rdfs:label ?pLabel . FILTER(LANG(?pLabel)="en")
    }} GROUP BY ?p ?pLabel ORDER BY DESC(?n) LIMIT {limit}
    """
    r = requests.get(ENDPOINT, params={"query": query}, headers=HEADERS, timeout=120)
    r.raise_for_status()
    for row in r.json()["results"]["bindings"]:
        print(f'{int(row["n"]["value"]):>6}  {row["p"]["value"].split("/")[-1]:8} {row["pLabel"]["value"]}')

In [19]:
top_properties_for_class("Q180684")   # properties commonly used on "conflict" instances

HTTPError: 504 Server Error: Gateway Timeout for url: https://query.wikidata.org/sparql?query=%0A++++SELECT+%3Fp+%3FpLabel+%28COUNT%28%2A%29+AS+%3Fn%29+WHERE+%7B%0A++++++%7B+SELECT+%3Fitem+WHERE+%7B+%3Fitem+wdt%3AP31+wd%3AQ180684+.+%7D+LIMIT+300+%7D%0A++++++%3Fitem+%3Fprop+%3Fval+.%0A++++++%3Fp+wikibase%3AdirectClaim+%3Fprop+%3B+rdfs%3Alabel+%3FpLabel+.+FILTER%28LANG%28%3FpLabel%29%3D%22en%22%29%0A++++%7D+GROUP+BY+%3Fp+%3FpLabel+ORDER+BY+DESC%28%3Fn%29+LIMIT+15%0A++++